# Guía 10 — Optimización de Hiperparámetros y Pipeline Final

> **Nota previa:** Sube `dataset_limpio.csv` dentro de `/content/crudos/` antes de ejecutar el notebook completo. Si el modelo de la Guía 09 (`modelo_candidato_proyecto_guia09.joblib`) existe en `/content/modelos/`, se cargará automáticamente; de lo contrario, se entrenará como referencia.

In [ ]:
# CREACIÓN DE CARPETAS
!mkdir -p crudos procesados documentos modelos
print("CARPETAS CREADAS")

In [ ]:
# IMPORTACIÓN DE LIBRERÍAS
from pathlib import Path
import sys
import time
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, ParameterGrid, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("LIBRERÍAS IMPORTADAS CORRECTAMENTE")

In [ ]:
# DEFINICIÓN DE RUTAS
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR / "crudos"
DATA_PROCESSED = BASE_DIR / "procesados"
DATA_DOCS = BASE_DIR / "documentos"
DATA_MODELS = BASE_DIR / "modelos"

for carpeta in [DATA_RAW, DATA_PROCESSED, DATA_DOCS, DATA_MODELS]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("RUTAS CONFIGURADAS!!")

In [ ]:
# CARGA DEL DATASET LIMPIO
limpio = DATA_RAW / "dataset_limpio.csv"
df = pd.read_csv(limpio)
print(df.columns.tolist())
print(f"Filas: {len(df)}")

In [ ]:
# PREPARACIÓN DE SERIE DIARIA, FEATURES Y SPLIT TEMPORAL 80/20
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])
df["festival_bin"] = df["festival"].map({"Yes": 1, "No": 0})

df_diario = (
    df.groupby("fecha_pedido")
    .agg(
        ventas_por_dia=("precio_pedido", "sum"),
        festival=("festival_bin", "max")
    )
    .reset_index()
    .sort_values("fecha_pedido")
    .reset_index(drop=True)
)

# Variables de rezago y media móvil sin fuga de datos
df_diario["lag_1"] = df_diario["ventas_por_dia"].shift(1)
df_diario["lag_7"] = df_diario["ventas_por_dia"].shift(7)
df_diario["media_movil_7"] = (
    df_diario["ventas_por_dia"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

# Dataset de modelado sin NaN en lags
df_modelo = df_diario.dropna(subset=["lag_1", "lag_7", "media_movil_7"]).copy()
df_modelo["dia_semana"] = df_modelo["fecha_pedido"].dt.dayofweek
df_modelo["mes"] = df_modelo["fecha_pedido"].dt.month

features = [
    "dia_semana",
    "mes",
    "festival",
    "lag_1",
    "lag_7",
    "media_movil_7"
]

X = df_modelo[features]
y = df_modelo["ventas_por_dia"]

# Split temporal cronológico 80/20
corte = int(len(df_modelo) * 0.80)
X_train = X.iloc[:corte]
X_test = X.iloc[corte:]
y_train = y.iloc[:corte]
y_test = y.iloc[corte:]
fecha_test = df_modelo["fecha_pedido"].iloc[corte:]

print("PREPARACIÓN COMPLETADA")
print("=" * 60)
print(f"Dataset de modelado: {len(df_modelo)} filas")
print(f"Entrenamiento: {len(X_train)} filas ({df_modelo['fecha_pedido'].iloc[0].date()} a {df_modelo['fecha_pedido'].iloc[corte - 1].date()})")
print(f"Prueba:        {len(X_test)} filas ({df_modelo['fecha_pedido'].iloc[corte].date()} a {df_modelo['fecha_pedido'].iloc[-1].date()})")

# Recuperar modelo candidato de Guía 09 (o entrenarlo si se corre de forma independiente)
nombre_candidato = "Regresión lineal"
ruta_modelo_g9 = DATA_MODELS / "modelo_candidato_proyecto_guia09.joblib"

if ruta_modelo_g9.exists():
    modelo_candidato = joblib.load(ruta_modelo_g9)
    print(f"\nModelo candidato cargado desde: {ruta_modelo_g9.name}")
else:
    modelo_candidato = LinearRegression()
    modelo_candidato.fit(X_train, y_train)
    print("\nModelo candidato (Regresión lineal) entrenado como referencia.")

In [ ]:
# 10.2 Comprobación del contexto de las fases anteriores
variables_requeridas = [
    "df_modelo", "features", "X_train", "X_test",
    "y_train", "y_test", "fecha_test",
    "modelo_candidato", "nombre_candidato",
    "BASE_DIR", "DATA_DOCS", "DATA_PROCESSED", "DATA_MODELS"
]

faltantes = [nombre for nombre in variables_requeridas if nombre not in globals()]
if faltantes:
    raise NameError(
        "Faltan variables requeridas para la Fase 10: "
        + ", ".join(faltantes)
    )

print("CONTEXTO DE LAS FASES 8 Y 9")
print("=" * 65)
print(f"Dataset de modelado: {len(df_modelo)} filas")
print(f"Variables predictoras: {features}")
print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba: {X_test.shape[0]} registros")
print(f"Modelo seleccionado en Guía 9: {nombre_candidato}")
print(f"Versión de scikit-learn: {sklearn.__version__}")

if nombre_candidato != "Regresión lineal":
    raise ValueError(
        "Esta adaptación fue construida para el resultado actual de tu Guía 9 "
        "(Regresión lineal). El candidato actual es: " + str(nombre_candidato)
    )

# Confirmación del orden temporal: el conjunto de prueba debe venir después del entrenamiento.
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert len(X_test) == len(fecha_test)
assert X_train.index.max() < X_test.index.min(), (
    "El corte no respeta el orden temporal. Revisa la preparación."
)

print("\nVerificación temporal correcta: prueba posterior a entrenamiento.")

In [ ]:
# 10.3 Calcular referencias en el mismo holdout
baseline_g10 = DummyRegressor(strategy="mean")
baseline_g10.fit(X_train, y_train)
pred_baseline_g10 = baseline_g10.predict(X_test)

pred_guia9_g10 = modelo_candidato.predict(X_test)

mae_baseline_g10 = mean_absolute_error(y_test, pred_baseline_g10)
rmse_baseline_g10 = mean_squared_error(y_test, pred_baseline_g10) ** 0.5
r2_baseline_g10 = r2_score(y_test, pred_baseline_g10)

mae_guia9_g10 = mean_absolute_error(y_test, pred_guia9_g10)
rmse_guia9_g10 = mean_squared_error(y_test, pred_guia9_g10) ** 0.5
r2_guia9_g10 = r2_score(y_test, pred_guia9_g10)

print("MÉTRICAS DE REFERENCIA — MISMO CONJUNTO DE PRUEBA")
print("=" * 70)
print(
    f"Baseline → MAE: ${mae_baseline_g10:,.2f} | "
    f"RMSE: ${rmse_baseline_g10:,.2f} | R²: {r2_baseline_g10:.4f}"
)
print(
    f"Guía 9   → MAE: ${mae_guia9_g10:,.2f} | "
    f"RMSE: ${rmse_guia9_g10:,.2f} | R²: {r2_guia9_g10:.4f}"
)

In [ ]:
# 10.4 Construcción del Pipeline
pipeline_g10 = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler()),
    ("modelo", Ridge())
])

print("PIPELINE DE OPTIMIZACIÓN")
print("=" * 65)
print(pipeline_g10)

In [ ]:
# 10.5 Cuadrícula de hiperparámetros
param_grid_g10 = {
    "modelo__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "modelo__fit_intercept": [True, False]
}

combinaciones_g10 = list(ParameterGrid(param_grid_g10))

print(f"Combinaciones de hiperparámetros: {len(combinaciones_g10)}")
print("Folds temporales: 5")
print(f"Entrenamientos esperados: {len(combinaciones_g10) * 5}")
print("\nParámetros que se evaluarán:")
for i, combinacion in enumerate(combinaciones_g10, start=1):
    print(f"{i:02d}. {combinacion}")

In [ ]:
# 10.6 Configurar TimeSeriesSplit y GridSearchCV
cv_g10 = TimeSeriesSplit(n_splits=5)

print("FOLDS DE VALIDACIÓN TEMPORAL")
print("=" * 75)
for fold, (idx_train, idx_val) in enumerate(cv_g10.split(X_train), start=1):
    print(
        f"Fold {fold}: "
        f"entrenamiento {idx_train[0]}–{idx_train[-1]} | "
        f"validación {idx_val[0]}–{idx_val[-1]}"
    )

busqueda_g10 = GridSearchCV(
    estimator=pipeline_g10,
    param_grid=param_grid_g10,
    scoring="neg_mean_absolute_error",
    cv=cv_g10,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    error_score="raise",
    verbose=1
)

print("\nGridSearchCV configurado correctamente.")

In [ ]:
# 10.7 Ejecutar la búsqueda usando únicamente el entrenamiento
inicio_g10 = time.perf_counter()
busqueda_g10.fit(X_train, y_train)
tiempo_total_g10 = time.perf_counter() - inicio_g10

print("\n" + "=" * 65)
print(f"Tiempo total de búsqueda: {tiempo_total_g10:.2f} segundos")
print(f"Mejores hiperparámetros: {busqueda_g10.best_params_}")
print(
    "MAE promedio en validación cruzada: "
    f"${-busqueda_g10.best_score_:,.2f}"
)
print("=" * 65)

In [ ]:
# 10.8 Tabla completa de resultados de la búsqueda
resultados_grid_g10 = pd.DataFrame(busqueda_g10.cv_results_)

columnas_grid_g10 = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "mean_fit_time",
    "params"
]

tabla_grid_g10 = resultados_grid_g10[columnas_grid_g10].copy()
tabla_grid_g10["MAE_validacion"] = -tabla_grid_g10["mean_test_score"]
tabla_grid_g10["MAE_entrenamiento"] = -tabla_grid_g10["mean_train_score"]
tabla_grid_g10["Brecha_MAE"] = (
    tabla_grid_g10["MAE_validacion"] -
    tabla_grid_g10["MAE_entrenamiento"]
)

tabla_grid_g10 = (
    tabla_grid_g10
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

display(
    tabla_grid_g10[
        [
            "rank_test_score", "MAE_validacion",
            "std_test_score", "MAE_entrenamiento",
            "Brecha_MAE", "mean_fit_time", "params"
        ]
    ].round(3)
)

In [ ]:
# 10.9 Mostrar las 10 mejores configuraciones
top10_g10 = tabla_grid_g10.head(10).copy()

print("TOP 10 CONFIGURACIONES")
print("=" * 80)
display(
    top10_g10[
        [
            "rank_test_score",
            "MAE_validacion",
            "std_test_score",
            "MAE_entrenamiento",
            "Brecha_MAE",
            "mean_fit_time",
            "params"
        ]
    ].round(3)
)

In [ ]:
# 10.10 Gráfico del Top 10 de configuraciones
top_g10 = tabla_grid_g10.nsmallest(10, "MAE_validacion").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    range(len(top_g10)),
    top_g10["MAE_validacion"],
    yerr=top_g10["std_test_score"],
    capsize=4,
    edgecolor="black"
)

ax.set_xticks(range(len(top_g10)))
ax.set_xticklabels([f"C{i+1}" for i in range(len(top_g10))])
ax.set_ylabel("MAE de validación ($)")
ax.set_title("Top 10 configuraciones — optimización Ridge", fontweight="bold")
ax.grid(alpha=0.3, axis="y", linestyle="--")

plt.tight_layout()
plt.show()

In [ ]:
# 10.11 Evaluar el mejor Pipeline en el holdout final
modelo_optimizado_g10 = busqueda_g10.best_estimator_
pred_optimizado_g10 = modelo_optimizado_g10.predict(X_test)

mae_optimizado_g10 = mean_absolute_error(y_test, pred_optimizado_g10)
rmse_optimizado_g10 = mean_squared_error(y_test, pred_optimizado_g10) ** 0.5
r2_optimizado_g10 = r2_score(y_test, pred_optimizado_g10)

print("MODELO OPTIMIZADO — CONJUNTO DE PRUEBA")
print("=" * 70)
print(f"MAE:  ${mae_optimizado_g10:,.2f}")
print(f"RMSE: ${rmse_optimizado_g10:,.2f}")
print(f"R²:   {r2_optimizado_g10:.4f}")
print(f"Mejores parámetros: {busqueda_g10.best_params_}")

In [ ]:
# 10.12 Tabla comparativa antes y después de optimizar
comparacion_guia10 = pd.DataFrame([
    {
        "Modelo": "Baseline",
        "MAE": mae_baseline_g10,
        "RMSE": rmse_baseline_g10,
        "R2": r2_baseline_g10
    },
    {
        "Modelo": "Guía 9 — Regresión lineal",
        "MAE": mae_guia9_g10,
        "RMSE": rmse_guia9_g10,
        "R2": r2_guia9_g10
    },
    {
        "Modelo": "Guía 10 — Ridge optimizado",
        "MAE": mae_optimizado_g10,
        "RMSE": rmse_optimizado_g10,
        "R2": r2_optimizado_g10
    }
])

comparacion_guia10["Mejora_vs_baseline_%"] = (
    (mae_baseline_g10 - comparacion_guia10["MAE"]) /
    mae_baseline_g10 * 100
)

comparacion_guia10["Mejora_vs_Guia9_%"] = (
    (mae_guia9_g10 - comparacion_guia10["MAE"]) /
    mae_guia9_g10 * 100
)

print("COMPARACIÓN DE MODELOS")
print("=" * 90)
display(comparacion_guia10.round(2))

In [ ]:
# 10.13 Gráfico comparativo de MAE
fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar(
    comparacion_guia10["Modelo"],
    comparacion_guia10["MAE"],
    edgecolor="black"
)

for barra, valor in zip(barras, comparacion_guia10["MAE"]):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height(),
        f"${valor:,.2f}",
        ha="center",
        va="bottom"
    )

ax.set_ylabel("MAE en el conjunto de prueba ($)")
ax.set_title("Comparación de error — Baseline, Guía 9 y Guía 10")
ax.tick_params(axis="x", rotation=15)
ax.grid(alpha=0.3, axis="y", linestyle="--")

plt.tight_layout()
plt.show()

In [ ]:
# 10.14 Predicciones, residuos y errores
errores_guia10 = pd.DataFrame({
    "fecha": pd.to_datetime(fecha_test).values,
    "real": np.asarray(y_test),
    "predicho": pred_optimizado_g10
})

errores_guia10["error"] = (
    errores_guia10["real"] - errores_guia10["predicho"]
)
errores_guia10["error_abs"] = errores_guia10["error"].abs()

print("ESTADÍSTICAS DE ERROR")
print("=" * 65)
print(f"Error promedio: ${errores_guia10['error'].mean():,.2f}")
print(f"Error mínimo:   ${errores_guia10['error'].min():,.2f}")
print(f"Error máximo:   ${errores_guia10['error'].max():,.2f}")

print("\nTOP 10 ERRORES ABSOLUTOS")
display(
    errores_guia10
    .nlargest(10, "error_abs")
    .round(2)
)

In [ ]:
# 10.15 Gráfico temporal: ventas reales vs. predicciones optimizadas
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    fecha_test,
    np.asarray(y_test),
    label="Ventas reales",
    linewidth=2,
    marker="o",
    markersize=3
)

ax.plot(
    fecha_test,
    pred_optimizado_g10,
    label="Predicción optimizada",
    linewidth=2,
    linestyle="--",
    marker="s",
    markersize=3
)

ax.set_title("Ventas reales vs. predicción — Modelo optimizado")
ax.set_xlabel("Fecha")
ax.set_ylabel("Ventas por día ($)")
ax.legend()
ax.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.show()

In [ ]:
# 10.16 Exportación de artefactos
for carpeta in [DATA_DOCS, DATA_PROCESSED, DATA_MODELS]:
    carpeta.mkdir(parents=True, exist_ok=True)

ruta_grid_g10 = DATA_DOCS / "gridsearch_resultados_proyecto_guia10.csv"
ruta_top10_g10 = DATA_DOCS / "top10_configuraciones_proyecto_guia10.csv"
ruta_comparacion_g10 = DATA_DOCS / "comparacion_antes_despues_proyecto_guia10.csv"
ruta_predicciones_g10 = DATA_PROCESSED / "predicciones_proyecto_guia10.csv"
ruta_modelo_g10 = DATA_MODELS / "modelo_optimizado_proyecto_guia10.joblib"

resultados_grid_g10.to_csv(ruta_grid_g10, index=False, encoding="utf-8")
top10_g10.to_csv(ruta_top10_g10, index=False, encoding="utf-8")
comparacion_guia10.to_csv(ruta_comparacion_g10, index=False, encoding="utf-8")
errores_guia10.to_csv(ruta_predicciones_g10, index=False, encoding="utf-8")
joblib.dump(modelo_optimizado_g10, ruta_modelo_g10)

print("ARTEFACTOS EXPORTADOS")
print("=" * 75)
for ruta in [
    ruta_grid_g10,
    ruta_top10_g10,
    ruta_comparacion_g10,
    ruta_predicciones_g10,
    ruta_modelo_g10
]:
    print(f"  {ruta}")

In [ ]:
# 10.17 Registrar metadata de la optimización
metadata_guia10 = {
    "fecha_ejecucion": pd.Timestamp.now().isoformat(),
    "python": sys.version.split()[0],
    "scikit_learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "dataset": "dataset_limpio existente reutilizado en fases 8 y 9",
    "filas_dataset_modelo": len(df_modelo),
    "n_train": len(X_train),
    "n_test": len(X_test),
    "n_features": len(features),
    "features": ", ".join(features),
    "modelo_candidato_guia9": nombre_candidato,
    "modelo_optimizado_guia10": "Ridge",
    "n_combinaciones": len(combinaciones_g10),
    "n_folds": cv_g10.get_n_splits(),
    "tiempo_busqueda_segundos": round(tiempo_total_g10, 2),
    "mejores_parametros": str(busqueda_g10.best_params_),
    "mae_baseline": round(mae_baseline_g10, 4),
    "mae_guia9": round(mae_guia9_g10, 4),
    "mae_optimizado": round(mae_optimizado_g10, 4),
    "mejora_vs_guia9_pct": round(
        (mae_guia9_g10 - mae_optimizado_g10) / mae_guia9_g10 * 100, 4
    )
}

ruta_metadata_g10 = DATA_DOCS / "metadata_proyecto_guia10.csv"
pd.DataFrame([metadata_guia10]).to_csv(
    ruta_metadata_g10,
    index=False,
    encoding="utf-8"
)

print("METADATA DEL EXPERIMENTO")
print("=" * 75)
for clave, valor in metadata_guia10.items():
    print(f"{clave}: {valor}")
print(f"\nMetadata guardada en: {ruta_metadata_g10}")

In [ ]:
# 10.18 Auditoría final de archivos
archivos_esperados_g10 = [
    ruta_grid_g10,
    ruta_top10_g10,
    ruta_comparacion_g10,
    ruta_predicciones_g10,
    ruta_modelo_g10,
    ruta_metadata_g10
]

print("AUDITORÍA FINAL — GUÍA 10")
print("=" * 85)

todo_ok_g10 = True
for ruta in archivos_esperados_g10:
    existe = ruta.exists()
    peso = f"{ruta.stat().st_size / 1024:.1f} KB" if existe else "—"
    estado = "OK" if existe else "FALTA"
    print(f"{estado:<8} {ruta.name:<55} {peso:>10}")
    todo_ok_g10 = todo_ok_g10 and existe

print("\n" + (
    "Todos los artefactos fueron generados correctamente."
    if todo_ok_g10 else
    "Faltan artefactos. Revisa las celdas de exportación."
))

In [ ]:
# 10.19 Crear ZIP con los artefactos de la Guía 10
ruta_zip_g10 = BASE_DIR / "artefactos_proyecto_guia10.zip"

with zipfile.ZipFile(ruta_zip_g10, "w", zipfile.ZIP_DEFLATED) as zipf:
    for ruta in archivos_esperados_g10:
        if ruta.exists():
            zipf.write(ruta, arcname=ruta.name)

print(f"Paquete creado: {ruta_zip_g10}")
print(f"Tamaño: {ruta_zip_g10.stat().st_size / 1024:.1f} KB")

# En Google Colab se iniciará la descarga. Fuera de Colab, el ZIP
# permanecerá guardado en la carpeta raíz del proyecto.
try:
    from google.colab import files
    files.download(str(ruta_zip_g10))
except Exception:
    print("La descarga automática solo está disponible en Google Colab.")

In [ ]:
# 10.20 Resumen ejecutivo calculado con los resultados reales
mejora_final_g10 = (
    (mae_guia9_g10 - mae_optimizado_g10) / mae_guia9_g10 * 100
)

print("=" * 80)
print("RESUMEN EJECUTIVO — FASE 10: OPTIMIZACIÓN DE HIPERPARÁMETROS")
print("=" * 80)
print("Modelo de la Guía 9: Regresión lineal")
print("Modelo optimizado: Ridge")
print(f"Mejores hiperparámetros: {busqueda_g10.best_params_}")
print(f"MAE de Guía 9: ${mae_guia9_g10:,.2f}")
print(f"MAE optimizado: ${mae_optimizado_g10:,.2f}")
print(f"Mejora respecto a Guía 9: {mejora_final_g10:.2f}%")

if mejora_final_g10 > 0:
    print("\nConclusión: Ridge redujo el MAE respecto al modelo de la Guía 9.")
elif np.isclose(mejora_final_g10, 0, atol=0.01):
    print("\nConclusión: Ridge obtuvo un desempeño prácticamente igual al de la Guía 9.")
else:
    print("\nConclusión: Ridge no superó el MAE de la Guía 9 en el conjunto de prueba.")

print("\nLa conclusión se basa en el conjunto de prueba temporal reservado.")
print("=" * 80)